# Inline payloads for Amazon SageMaker Asynchronous Inference

Amazon SageMaker Asynchronous Inference lets you queue inference requests and process them in the background — ideal for large payloads, long-running inference, and bursty traffic that can tolerate seconds-to-minutes latency. Until recently, every invocation required a two-step pattern: **upload the input to Amazon S3, then call the endpoint with the S3 URI** (`InputLocation`).

With inline payload support, `invoke_endpoint_async` now accepts a **`Body`** parameter — the same parameter name the synchronous `invoke_endpoint` API already uses. For payloads up to **128,000 bytes**, you can send the request body directly in the API call — no input upload to S3, no input bucket to manage, no `s3:PutObject` grant on the *input* path, and no stale input objects to clean up.

> **Note on the output path.** Asynchronous inference always writes the *result* to the S3 location you configure (`AsyncInferenceConfig.OutputConfig.S3OutputPath`). Inline `Body` removes the **input** upload — it does not change the output path. You still configure an output S3 location, and the endpoint's execution role still needs `s3:PutObject` there to store results.

This notebook:

1. Deploys a small, inexpensive asynchronous endpoint.
2. Invokes it the **new** way — inline `Body`.
3. Invokes it the **classic** way — `InputLocation` (S3) — to show full backward compatibility.
4. Demonstrates the validation behavior (mutual exclusivity, size limit).
5. Cleans up all resources.

> **Prerequisite.** The inline `Body` parameter on `InvokeEndpointAsync` requires **boto3/botocore ≥ 1.43.29**. The setup section below pins that floor and also detects the capability at runtime, failing early with an upgrade hint if your installed SDK is too old.

> **Where to run this.** It runs as-is in **Amazon SageMaker Studio** (the execution role and credentials are ambient). It also runs in **Google Colab or on a laptop** — there's no SageMaker-managed role there, so first configure AWS credentials and set the `SAGEMAKER_ROLE_ARN` environment variable to an existing SageMaker execution role ARN. Either way the endpoint runs in your AWS account and incurs charges until the cleanup cell.

## 1. Setup

The inline `Body` parameter on `invoke_endpoint_async` requires **boto3/botocore ≥ 1.43.29** (the release that introduced it). The cell below pins that floor and installs the SageMaker SDK; the following cell verifies the capability at runtime as a safety net.

In [ ]:
# The inline `Body` parameter on InvokeEndpointAsync first ships in boto3/botocore
# 1.43.29 (verified: 1.43.28 does not expose it). Pin that floor, and ensure the
# SageMaker SDK is present. We upgrade only boto3/botocore here (the packages that
# carry the new parameter); the next cell still verifies the capability at runtime as
# a safety net.
%pip install -Uq "boto3>=1.43.29" "botocore>=1.43.29"
%pip install -q "sagemaker>=2.198.0"

In [ ]:
import json
import os
import time
import uuid

import boto3
import sagemaker
from sagemaker.huggingface import HuggingFaceModel

sess = sagemaker.Session()
region = sess.boto_region_name
bucket = sess.default_bucket()
prefix = "async-inline-payload-demo"

# Resolve the SageMaker execution role (the role SageMaker assumes to run the endpoint).
# Precedence:
#   1. SAGEMAKER_ROLE_ARN env var, if set — always wins. Use this anywhere outside
#      SageMaker Studio (Google Colab, a laptop, CloudShell, an EC2 instance-profile
#      session, etc.), where the *caller's* identity is NOT a valid SageMaker
#      execution role.
#   2. Otherwise, the ambient Studio / Notebook-Instance execution role.
# This ordering matters: outside Studio, get_execution_role() may return the CALLER's
# role (e.g. an admin or instance-profile role) that SageMaker cannot assume — its
# trust policy doesn't allow sagemaker.amazonaws.com — which surfaces later as an
# endpoint "Failed" with an "execution role ARN is invalid" message. Setting
# SAGEMAKER_ROLE_ARN to a real SageMaker execution role avoids that.
role = os.environ.get("SAGEMAKER_ROLE_ARN")
if not role:
    try:
        role = sagemaker.get_execution_role()
    except Exception as e:
        raise RuntimeError(
            "No execution role available. Set the SAGEMAKER_ROLE_ARN environment "
            "variable to an existing SageMaker execution role ARN (and configure AWS "
            "credentials) before running this notebook outside SageMaker Studio."
        ) from e

sm_client = boto3.client("sagemaker", region_name=region)
smr_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"sagemaker version : {sagemaker.__version__}")
print(f"boto3 version     : {boto3.__version__}")
print(f"region            : {region}")
print(f"role              : {role}")
print(f"bucket            : s3://{bucket}/{prefix}")

In [ ]:
# Confirm the running SDK actually exposes the inline-body parameter before going further.
# This is a cheap guard so the notebook fails loudly (here) rather than mid-demo.
# The parameter is named "Body" (same as the synchronous InvokeEndpoint API); we also
# accept "InputBody" defensively in case an SDK build names it differently.
members = smr_client.meta.service_model.operation_model("InvokeEndpointAsync").input_shape.members
inline_param = next((m for m in ("Body", "InputBody") if m in members), None)

if inline_param is None:
    raise RuntimeError(
        "This boto3/botocore build does not expose an inline-body parameter on "
        "InvokeEndpointAsync. Upgrade the SDK (see the pip cell above) and restart "
        "the kernel.\n"
        f"Available InvokeEndpointAsync input members: {sorted(members)}"
    )

print(f"Inline-body parameter detected: {inline_param!r}")
print("Available InvokeEndpointAsync input members:")
print(json.dumps(sorted(members), indent=2))

## 2. Deploy a small asynchronous endpoint

We deploy a small Hugging Face text-classification model (`distilbert-base-uncased-finetuned-sst-2-english`, ~67M parameters) on a CPU instance (`ml.m5.xlarge`). It is cheap, downloads in well under a minute, and returns compact JSON — perfect for demonstrating the request *surface* rather than model performance.

The only thing that makes an endpoint "asynchronous" is the **`AsyncInferenceConfig`** on the endpoint configuration (an S3 output location). The model container itself is unchanged — it receives the same HTTP request whether the payload arrived inline or from S3.

In [ ]:
# A small, CPU-friendly text-classification model from the Hugging Face Hub.
hub = {
    "HF_MODEL_ID": "distilbert-base-uncased-finetuned-sst-2-english",
    "HF_TASK": "text-classification",
}

# Version pins map to the CPU Hugging Face inference DLC:
#   pytorch-inference:2.1.0-transformers4.37.0-cpu-py310-ubuntu22.04
# (the only py310 CPU Hugging Face inference image published at time of writing).
hf_model = HuggingFaceModel(
    env=hub,
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
    sagemaker_session=sess,
)

# Build the container/model definition WITHOUT deploying, so we can attach an
# AsyncInferenceConfig on the endpoint config ourselves.
container_def = hf_model.prepare_container_def(instance_type="ml.m5.xlarge")
model_name = sagemaker.utils.name_from_base("async-inline-distilbert")
sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer=container_def,
)
print(f"Created model: {model_name}")

In [ ]:
# Endpoint config with AsyncInferenceConfig -> this is what makes it an async endpoint.
endpoint_config_name = sagemaker.utils.name_from_base("async-inline-cfg")
async_output_path = f"s3://{bucket}/{prefix}/async-output"

sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": "ml.m5.xlarge",
            "InitialInstanceCount": 1,
        }
    ],
    AsyncInferenceConfig={
        "OutputConfig": {
            "S3OutputPath": async_output_path,
            # No S3FailurePath / no inline-response config here, to keep the demo simple.
        }
    },
)
print(f"Created endpoint config: {endpoint_config_name}")
print(f"Async output path      : {async_output_path}")

In [ ]:
endpoint_name = sagemaker.utils.name_from_base("async-inline-ep")
sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"Creating endpoint: {endpoint_name} ... (typically 5-8 minutes)")

waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=endpoint_name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
print(f"Endpoint in service: {endpoint_name}")

## 3. A small helper to fetch async results

`invoke_endpoint_async` returns immediately with an `OutputLocation` (an S3 URI). The model result is written there once inference completes — or, on a processing error, to the `FailureLocation`. The helper below takes the full invoke response and polls both, so a success returns the parsed JSON and a failure raises promptly instead of hanging. Both invocation styles below reuse it — the result-handling path is identical regardless of how the input was sent.

In [ ]:
def _split_s3_uri(s3_uri):
    if not s3_uri.startswith("s3://"):
        raise ValueError(f"Not an S3 URI: {s3_uri!r}")
    b, k = s3_uri[len("s3://"):].split("/", 1)
    return b, k


def wait_for_async_output(invoke_response, timeout_s=300, poll_s=5):
    """Poll the async result until it appears; return parsed JSON.

    Pass the full invoke_endpoint_async response. The result is written to
    OutputLocation on success, or to FailureLocation on model/processing error;
    we poll both so a failure surfaces promptly instead of hanging until timeout.
    """
    out_bucket, out_key = _split_s3_uri(invoke_response["OutputLocation"])
    fail_uri = invoke_response.get("FailureLocation")
    fail = _split_s3_uri(fail_uri) if fail_uri else None

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            obj = s3_client.get_object(Bucket=out_bucket, Key=out_key)
            return json.loads(obj["Body"].read().decode("utf-8"))
        except s3_client.exceptions.NoSuchKey:
            pass
        if fail:
            try:
                err_obj = s3_client.get_object(Bucket=fail[0], Key=fail[1])
                raise RuntimeError(
                    "Async inference failed; FailureLocation contains: "
                    + err_obj["Body"].read().decode("utf-8", "replace")
                )
            except s3_client.exceptions.NoSuchKey:
                pass
        time.sleep(poll_s)
    raise TimeoutError(
        f"No async result at {invoke_response['OutputLocation']} after {timeout_s}s"
    )

## 4. The new way — inline `Body`

Send the payload directly in the API call. No S3 client, no upload, no input bucket.

In [ ]:
payload = json.dumps({"inputs": "Inline payloads make async inference so much simpler!"}).encode("utf-8")

# `inline_param` was detected in the setup cell — it is "Body" on current SDKs.
kwargs = {
    "EndpointName": endpoint_name,
    "ContentType": "application/json",
    inline_param: payload,   # <-- the whole point: payload travels inline, no S3 upload
}

start = time.time()
response = smr_client.invoke_endpoint_async(**kwargs)
submit_latency = time.time() - start

print(f"Submitted inline in {submit_latency:.3f}s (no S3 upload)")
print(f"InferenceId    : {response.get('InferenceId')}")
print(f"OutputLocation : {response['OutputLocation']}")

result = wait_for_async_output(response)
print("\nModel result:")
print(json.dumps(result, indent=2))

## 5. The classic way — `InputLocation` (S3)

For comparison and to prove backward compatibility: upload the payload to S3 first, then pass its URI as `InputLocation`. This is exactly how async inference worked before inline support — and it still works unchanged.

In [ ]:
# 1. Upload the payload to S3 (the extra step inline payloads remove).
input_key = f"{prefix}/async-input/{uuid.uuid4()}.json"
s3_client.put_object(
    Bucket=bucket,
    Key=input_key,
    Body=json.dumps({"inputs": "The classic S3 InputLocation path still works."}).encode("utf-8"),
)
input_location = f"s3://{bucket}/{input_key}"
print(f"Uploaded input to: {input_location}")

# 2. Invoke with InputLocation (no inline Body).
start = time.time()
response_s3 = smr_client.invoke_endpoint_async(
    EndpointName=endpoint_name,
    ContentType="application/json",
    InputLocation=input_location,
)
submit_latency_s3 = time.time() - start

print(f"Submitted via S3 in {submit_latency_s3:.3f}s (upload + invoke)")
print(f"OutputLocation : {response_s3['OutputLocation']}")

result_s3 = wait_for_async_output(response_s3)
print("\nModel result:")
print(json.dumps(result_s3, indent=2))

Both calls hit the same endpoint, ran the same model, and produced the same shape of result. The only difference is how the input got there.

## 6. Validation behavior

The service validates inline requests **synchronously** — you get an immediate error before any work is queued. Two rules to know:

1. **Mutual exclusivity** — you cannot set both `Body` and `InputLocation`. Provide exactly one.
2. **Size limit** — inline `Body` cannot exceed **128,000 bytes**. This is a *byte* limit on the raw payload, not a character count: for text, it's the UTF-8 byte length, so multibyte characters (most non-ASCII text, emoji) count as more than one byte each. A safe client-side pre-check is `len(payload_bytes) <= 128_000`.

> The exact error message strings are framework-generated and may differ slightly across SDK/service versions, so the cells below assert on the *exception type and behavior* rather than on an exact string. (The size limit is enforced by the service, not the SDK — so an oversized payload surfaces as a synchronous `ValidationError` from the API call below, not as a local SDK error.)

In [ ]:
from botocore.exceptions import ClientError

# 6a. Setting both Body and InputLocation is rejected synchronously.
both_kwargs = {
    "EndpointName": endpoint_name,
    "ContentType": "application/json",
    "InputLocation": input_location,
    inline_param: payload,
}
try:
    smr_client.invoke_endpoint_async(**both_kwargs)
    print("UNEXPECTED: the call was accepted. Check the SDK/service version.")
except ClientError as e:
    err = e.response["Error"]
    print("Rejected as expected when both Body and InputLocation are set:")
    print(f"  Code   : {err['Code']}")
    print(f"  Message: {err['Message']}")

In [ ]:
# 6b. An oversized inline payload (> 128,000 bytes) is rejected synchronously.
oversized = json.dumps({"inputs": "x" * 130_000}).encode("utf-8")
print(f"Oversized payload size: {len(oversized):,} bytes")

over_kwargs = {
    "EndpointName": endpoint_name,
    "ContentType": "application/json",
    inline_param: oversized,
}
try:
    smr_client.invoke_endpoint_async(**over_kwargs)
    print("UNEXPECTED: the call was accepted. Check the SDK/service version.")
except ClientError as e:
    err = e.response["Error"]
    print("Rejected as expected for payload > 128,000 bytes:")
    print(f"  Code   : {err['Code']}")
    print(f"  Message: {err['Message']}")

## 7. When to use each approach

| Scenario | Recommended approach |
|---|---|
| Payload &le; 128,000 bytes (JSON prompts, structured data) | **Inline `Body`** — simpler, faster, cheaper |
| Payload > 128,000 bytes (images, audio, large documents) | **`InputLocation`** — upload to S3 first |
| Mixed workload with variable payload sizes | Branch on size: `Body` for small, `InputLocation` for large |
| You need to retain inputs in S3 for audit/replay | **`InputLocation`** — keeps inputs in your bucket |

For the common case of small payloads, inline `Body` removes an entire network round-trip and an `s3:PutObject` per request on the **input** path, and lets you drop the **input** bucket, its lifecycle policies, and the IAM grant on the input path. The **output** path is unchanged: async inference still writes results to your configured `S3OutputPath`, so you keep an output bucket and the execution role's `s3:PutObject` there.

## 8. Clean up

Delete the endpoint, endpoint config, and model to stop incurring charges. Optionally remove the demo S3 objects.

In [ ]:
sm_client.delete_endpoint(EndpointName=endpoint_name)
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm_client.delete_model(ModelName=model_name)
print("Deleted endpoint, endpoint config, and model.")

# Optional: remove demo S3 objects (input + async output).
paginator = s3_client.get_paginator("list_objects_v2")
to_delete = []
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    to_delete.extend({"Key": o["Key"]} for o in page.get("Contents", []))
for i in range(0, len(to_delete), 1000):
    s3_client.delete_objects(Bucket=bucket, Delete={"Objects": to_delete[i:i + 1000]})
print(f"Deleted {len(to_delete)} demo object(s) under s3://{bucket}/{prefix}.")